# 04 - Deflacao dos custos pelo CPI do Canada (grao anual)

Converte os custos internos **nominais** (CAD) para **valores reais de dezembro/2025**
usando o **Consumer Price Index (CPI) all-items do Canada** (Statistics Canada, vetor
v41690973, base 2002=100). A correcao monetaria elimina o efeito da inflacao: assim,
diferencas observadas entre anos refletem mudancas **reais** no comportamento dos
custos de manutencao, e nao apenas a perda do poder de compra da moeda.

A deflacao e feita no grao **mensal** (cada OS pelo indice do seu mes) e depois
**somada por carreta x ano**, produzindo a variavel resposta do projeto:

> **Y = custo anual de manutencao por carreta (CAD/ano), em valores reais (dez/2025).**

Saida: `data/processed/base_anual_carreta_deflacionada.csv`.


In [1]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
TABLES = PROJECT_ROOT / "reports" / "tables"
FIGURES = PROJECT_ROOT / "reports" / "figures"
MES_BASE = pd.Timestamp("2025-12-01")


## 1. Carregar base anual, custos mensais e o indice CPI

O CPI e a **unica** serie externa do projeto (dado publico da Statistics Canada);
todo o restante vem da base consolidada `fato_wo_ml`.

In [2]:
base = pd.read_csv(DATA_PROCESSED / "base_anual_carreta.csv")
custo_mes = pd.read_csv(DATA_PROCESSED / "custo_carreta_mes.csv", parse_dates=["mes"])

cpi = pd.read_csv(DATA_RAW / "cpi_canada_statcan_2020_2025.csv", parse_dates=["ano_mes"])
cpi["indice_cpi"] = pd.to_numeric(cpi["indice_cpi"], errors="coerce")
cpi = cpi.sort_values("ano_mes").copy()
indice_base = float(cpi.loc[cpi["ano_mes"] == MES_BASE, "indice_cpi"].iloc[0])
cpi["fator_cpi_para_2025_12"] = indice_base / cpi["indice_cpi"]
cpi.to_csv(TABLES / "04_cpi_fatores.csv", index=False)
print(f"CPI dez/2025 = {indice_base} (base 2002=100) | meses: {len(cpi)}")
print(f"fator min={cpi['fator_cpi_para_2025_12'].min():.4f} (dez/2025) "
      f"max={cpi['fator_cpi_para_2025_12'].max():.4f} (mes mais antigo)")
print(f"inflacao acumulada 2020-01 -> 2025-12: "
      f"{(cpi['indice_cpi'].iloc[-1]/cpi['indice_cpi'].iloc[0]-1)*100:.1f}%")


CPI dez/2025 = 165.0 (base 2002=100) | meses: 72
fator min=0.9976 (dez/2025) max=1.2159 (mes mais antigo)
inflacao acumulada 2020-01 -> 2025-12: 20.6%


## 2. Deflacionar os custos mensais e somar por carreta x ano

`custo_real = custo_nominal * (CPI_dez2025 / CPI_mes)`. Cada carreta-ano recebe o
custo real total (Y) e o custo nominal total (para comparacao).

In [3]:
m = custo_mes.merge(cpi[["ano_mes", "fator_cpi_para_2025_12"]],
                     left_on="mes", right_on="ano_mes", how="left")
assert m["fator_cpi_para_2025_12"].isna().sum() == 0, "ha meses sem CPI"
m["custo_real_mes"] = m["custo_nominal_mes"] * m["fator_cpi_para_2025_12"]

y_real = (m.groupby(["id_carreta", "ano"])["custo_real_mes"].sum()
           .reset_index().rename(columns={"custo_real_mes": "custo_ano_real"}))
base = base.merge(y_real, on=["id_carreta", "ano"], how="left")
base["custo_ano_real"] = base["custo_ano_real"].fillna(0.0)   # anos ativos sem OS -> 0
print("Y (custo_ano_real) resumo:")
print(base["custo_ano_real"].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())
print("assimetria:", round(base['custo_ano_real'].skew(), 2),
      "| % zeros:", round((base['custo_ano_real'] == 0).mean()*100, 1))


Y (custo_ano_real) resumo:
count    49248.0
mean      1673.7
std       2400.7
min          0.0
25%        317.2
50%        812.5
75%       2009.0
90%       4221.8
99%      11820.4
max      62230.9
assimetria: 3.79 | % zeros: 3.2


## 3. Variaveis monetarias derivadas e historico defasado (em valor real)

`custo_medio_por_os_ano` e componente da propria resposta (Y = n_os x custo medio),
tratada com cautela na analise. `custo_ano_anterior` e `custo_acum_ate_ano_anterior`
usam apenas anos anteriores (anti-vazamento) e servem a modelagem preditiva.

In [4]:
base = base.sort_values(["id_carreta", "ano"]).reset_index(drop=True)
base["custo_medio_por_os_ano"] = np.where(base["n_os_ano"] > 0,
                                          base["custo_ano_real"] / base["n_os_ano"].replace(0, np.nan), 0.0)
g = base.groupby("id_carreta")
base["custo_ano_anterior"] = g["custo_ano_real"].shift(1)
base["custo_acum_ate_ano_anterior"] = g["custo_ano_real"].cumsum() - base["custo_ano_real"]
print(base[["id_carreta", "ano", "custo_ano_real", "custo_medio_por_os_ano",
            "custo_ano_anterior", "custo_acum_ate_ano_anterior"]].head(8).round(1).to_string(index=False))


 id_carreta  ano  custo_ano_real  custo_medio_por_os_ano  custo_ano_anterior  custo_acum_ate_ano_anterior
         19 2020           100.2                   100.2                 NaN                          0.0
         19 2021           608.2                   304.1               100.2                        100.2
         19 2022           100.1                    50.1               608.2                        708.4
         19 2023           211.7                   105.9               100.1                        808.5
         19 2024           171.8                    85.9               211.7                       1020.3
         19 2025           101.3                   101.3               171.8                       1192.1
         46 2020            23.8                    23.8                 NaN                          0.0
         46 2021            59.0                    59.0                23.8                         23.8


## 4. Comparacao anual nominal x real e validacao

In [5]:
anual = base.groupby("ano").agg(
    custo_nominal=("custo_ano_nominal", "sum"),
    custo_real=("custo_ano_real", "sum"),
    carretas=("id_carreta", "nunique"),
    custo_real_medio_carreta=("custo_ano_real", "mean"),
).reset_index()
anual["custo_nominal_mi"] = (anual["custo_nominal"] / 1e6).round(3)
anual["custo_real_mi"] = (anual["custo_real"] / 1e6).round(3)
anual.to_csv(TABLES / "04_comparacao_nominal_deflacionado.csv", index=False)

validacao = pd.DataFrame([
    {"checagem": "linhas", "valor": len(base)},
    {"checagem": "custo_nominal_total_mi", "valor": round(base['custo_ano_nominal'].sum()/1e6, 3)},
    {"checagem": "custo_real_total_mi", "valor": round(base['custo_ano_real'].sum()/1e6, 3)},
    {"checagem": "y_media_real", "valor": round(base['custo_ano_real'].mean(), 2)},
    {"checagem": "y_mediana_real", "valor": round(base['custo_ano_real'].median(), 2)},
    {"checagem": "y_pct_zero", "valor": round((base['custo_ano_real'] == 0).mean()*100, 2)},
    {"checagem": "fonte_cpi", "valor": "StatCan v41690973 - CPI all-items Canada (2002=100)"},
])
validacao.to_csv(TABLES / "04_validacao_deflacao.csv", index=False)
print(anual[["ano", "custo_nominal_mi", "custo_real_mi", "custo_real_medio_carreta"]].round(1).to_string(index=False))


 ano  custo_nominal_mi  custo_real_mi  custo_real_medio_carreta
2020               7.5            9.0                    1333.9
2021               8.4            9.8                    1288.2
2022              11.7           12.8                    1569.8
2023              15.0           15.8                    1799.2
2024              16.3           16.8                    1878.3
2025              18.1           18.2                    2025.8


In [6]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(anual["ano"], anual["custo_nominal_mi"], marker="o", label="Nominal (CAD corrente)")
ax.plot(anual["ano"], anual["custo_real_mi"], marker="s",
        label="Real (CAD de dez/2025, CPI Canada)")
ax.set_xlabel("Ano"); ax.set_ylabel("Custo interno total (CAD milhoes)")
ax.set_title("Custo anual de manutencao: nominal vs real (CPI Canada, base dez/2025)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(FIGURES / "04_nominal_vs_deflacionado.png", dpi=150)
plt.close(fig)
print("figura salva: reports/figures/04_nominal_vs_deflacionado.png")


figura salva: reports/figures/04_nominal_vs_deflacionado.png


## 5. Gravar base anual deflacionada

In [7]:
base.to_csv(DATA_PROCESSED / "base_anual_carreta_deflacionada.csv", index=False)
print("OK base anual deflacionada:", base.shape)
print("custo real total (mi CAD dez/2025):", round(base['custo_ano_real'].sum()/1e6, 2))


OK base anual deflacionada: (49248, 31)
custo real total (mi CAD dez/2025): 82.43
